## Settings

In [1]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [ ]:
## libraries
import sys
import pandas as pd
from pathlib import Path
from itertools import combinations

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories.registry import load_estimators
from src.evaluators.training import fit_predict_frontier
from src.evaluators.metrics import consensus_metrics
from src.evaluators.metrics import CONSENSUS_METRICS
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initalization

In [ ]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
data = load_processed_data()
models = load_estimators()

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

Original Data: 32 features, 25 observations


## Training

In [ ]:
## train models
if "frontiers" not in globals():
    frontiers = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        fit_result = fit_predict_frontier(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontiers[name] = fit_result["y_pred"]

## compute pairwise consensus
results_list = list()
for (i, y_true), (j, y_pred) in combinations(iterable = frontiers.items(), r = 2):
    
    ## calculate all metrics
    metrics = consensus_metrics(y_true = y_true, y_pred = y_pred)
    results_list.append({
        "model_i": i,
        "model_j": j,
        **metrics
    })

Training linear_quantile...
Training linear_convex...
Training linear_laws...
Training forest_quantile...
Training boosted_quantile...
Training xgboost_quantile...
Training neural_quantile...
Training neural_expectile...
Training neural_convex...


## Post-Processing

In [5]:
## convert to dataframe
results_data = pd.DataFrame(results_list)

## Learning Consensus Test
This section compares fitted model frontiers to each other rather than comparing each model to the target. Each learner is fit 30 times on the full dataset. Predictions are averaged across repeats before pairwise agreement is computed for every model pair.

### Reporting Convention
The table reports medians across all pairwise model comparisons. `rho` measures global rank agreement. `rbo` emphasizes agreement near the top of the ranking. `dcr` captures nonlinear dependence. `ci` is the geometric mean summary over `rho`, `rbo`, and `dcr`. Higher values indicate stronger cross-paradigm consensus.

In [6]:
## preview results
results_data[CONSENSUS_METRICS].median()

rho    0.961538
rbo    0.898660
dcr    0.961652
ci     0.940339
dtype: float64

Pairwise agreement is high across learning paradigms. Median Spearman rho is 0.962. Median RBO is 0.899. Median distance correlation is 0.962. Median CI is 0.940. Different model classes recover nearly the same fitted frontier ordering.